# tsfresh feature extracting method, not working because of RAM

In [ ]:
mask_labeled = df["target"] != -1  # Assuming -1 or some other value indicates unlabeled data
feature_columns = ['Current',"Voltage","Ah_Counter"]
df1 = df.loc[mask_labeled,feature_columns]

grouped_data = df.groupby(group_column)[feature_columns].apply(extract_features).reset_index()

df1["Index_Time"]=df1.index
#=df1["Current"].astype("float16")
#df1["target"]=df1["target"].astype(int)
#df1 = df1.stack()
time_domain_features = extract_features(timeseries_container=df1, column_id ='ID', column_sort='Index_Time',n_jobs=8)

In [ ]:
from tsfresh.utilities.dataframe_functions import impute

feature_columns = ['Current', 'Voltage',"Time","ID"]
group_column = ['group_cu','group_cu_procedure']
target_column = 'target'
X_train_list = []
y_train_list =[]

# Extract time domain features
time_domain_features = extract_features(df.loc[:,feature_columns], column_id ='ID', column_sort='Time', 
                                        default_fc_parameters=EfficientFCParameters())

# Extract derivative features
derivative_features = extract_features(df.loc[:,feature_columns].diff().dropna(axis=1, how='all'), 
                                    column_id= "ID", column_sort='Time',
                                    default_fc_parameters=EfficientFCParameters())

X_features = pd.concat([time_domain_features, derivative_features], axis=1)

X_group_raw = impute(X_features)

y_group = group['target'].values[0]

X_train_list.append(X_group_raw)
y_train_list.append(y_group)


X_train_array = np.array(X_train_list)
y_train_array = np.array(y_train_list)
y_train_categorical = to_categorical(y_train_array, num_classes) 
y = y_train_array

# # Split into labeled and unlabeled data
mask_labeled = y != -1  # Assuming -1 or some other value indicates unlabeled data
X_labeled = X_train_array[mask_labeled]
y_labeled =  to_categorical(y_train_array[mask_labeled],num_classes) 
X_unlabeled = X_train_array[~mask_labeled]



In [ ]:
print(df[['group_cu', 'group_cu_procedure']].drop_duplicates())

# LSTM Attempt, not working that well because of lack of data

In [ ]:

feature_columns = ['Current', 'Voltage']
group_column = ['group_cu','group_cu_procedure']
target_column = 'target'
X_train_list = []
y_train_list =[]
num_classes = len(df["target"].unique())-1 # Example number of classes


T_max = 8000    # Maximale Länge der Timeseries

groups = df.groupby(group_column)

for name, group in groups:
    X_group_raw = group[feature_columns].values  # Reshape zu (T, F)
    y_group = group['target'].values[0]

    X_train_list.append(X_group_raw)
    y_train_list.append(y_group)


X_group_padded  = [pad_sequences(X_train_list, maxlen=T_max, dtype='float32', padding='post', truncating='post')]
#y_group_padded  = [pad_sequences(y_train_list, maxlen=T_max, dtype='int32', padding='post', truncating='post')]

X_train_array = np.array(X_group_padded)[0]
y_train_array = np.array(y_train_list)
y_train_categorical = to_categorical(y_train_array, num_classes) 
y = y_train_array

# # Split into labeled and unlabeled data
mask_labeled = y != -1  # Assuming -1 or some other value indicates unlabeled data
X_labeled = X_train_array[mask_labeled]
y_labeled =  to_categorical(y_train_array[mask_labeled],num_classes) 
X_unlabeled = X_train_array[~mask_labeled]

from sklearn.model_selection import KFold

kf = KFold(n_splits=4)
for train_index, val_index in kf.split(X_labeled):
    X_train, X_val = X_labeled[train_index], X_labeled[val_index]
    y_train, y_val = y_labeled[train_index], y_labeled[val_index]

In [ ]:
y_val

In [ ]:
from keras.layers import Dense, LSTM, BatchNormalization

# Define LSTM model
model_multiclass = Sequential()
model_multiclass.add(Input(shape=(T_max, 2)))  # Use Input(shape) as the first layer
model_multiclass.add(LSTM(units=4, return_sequences=False))  # Add LSTM layer after Input
model_multiclass.add(Dense(num_classes, activation='softmax'))  # Output layer for multiclass classification
#model_multiclass.add(BatchNormalization())
# Compile the model with appropriate loss function and optimizer for multiclass classification
model_multiclass.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the modified model using X_labeled and y_train_categorical
model_multiclass.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20)


y_unlabeled_pred = model_multiclass.predict(X_unlabeled)


In [ ]:
y_unlabeled_pred[0]

In [ ]:

# For multiclass classification, find the index of the maximum value in each row
y_unlabeled_pred_classes = np.argmax(y_unlabeled_pred, axis=1)

In [ ]:
np.unique(y_unlabeled_pred_classes)

In [ ]:
df_List = []

for cell in List_Cell:
    savepath = working_path + '/cap_pulse_export/' + cell
    df_cell=read_cell(cell, rootpath, cell_dict)
    if df_cell is None:
        print(f"Stopping process: Unable to read cell {cell}")
        continue  # This will exit the for loop entirely
    else:
        df_List.append(df_cell)
        masks = create_masks(df_cell,program_info_catl, program_info_narada)
        df_cell, temp_results = process_filtered_data(df_cell, masks)

        #df_cell.to_parquet(savepath)
        #df_results.loc[df_cell["Name_SE"][0]]=temp_results


In [ ]:
plot_cell_data(df_List,Experiment_Sets,working_path,project_name_ahjo,"PulseResistance_py",save_figures=True)

In [ ]:
df_List[16].tail()

In [ ]:
for set, cell_list in Experiment_Sets.items():
    legend_items = []
    fig, ax = plt.subplots(clear=True)
    Ah_Nom = int(set.split("_")[-1])
    print(f"Ah_Nom: {Ah_Nom}")
    
    for df in df_List:
        if df["Name_SE"].iloc[0] in cell_list:
            df_plot = df.dropna(subset=['Capacity_py'])[0:3]
            cell_label = df_plot["Name_SE"].iloc[0]
            ax.scatter(df_plot["Ah_throughput"]/Ah_Nom, abs(df_plot['Capacity_py']), label=cell_label)
    
    handles, labels = ax.get_legend_handles_labels()
    
    # Sort handles and labels only if there's more than one item
    if len(handles) > 1:
        handles, labels = zip(*sorted(zip(handles, labels), key=lambda k: int(k[1])))
    
    ax.grid()
    plt.xlabel('Equivalente Vollzyklen')
    plt.ylabel('Gemessene Kapazität in Ah')
    plt.title(f'Welcome Check: Kapazitätstest für {set}')
    
    # Adjust legend based on number of items
    if len(handles) > 0:
        if len(handles) > 10:
            ncol = 2
        else:
            ncol = 1
        plt.legend(handles, labels, bbox_to_anchor=(1.02, 1), title='Siemens ID:', 
                   loc='upper left', fontsize='small', frameon=True, fancybox=False, 
                   framealpha=1, borderpad=0.5, labelspacing=0.2, handlelength=1, 
                   handletextpad=0.5, ncol=ncol)
        plt.subplots_adjust(right=0.8)
    else:
        print(f"No data to plot for set: {set}")

    plt.show()
    plt.savefig(working_path+"/cap_pulse_figure/"+set+"-PROJECT-"+project_name_ahjo+"_capacity.png", dpi=300)

In [ ]:
df_results.to_parquet(working_path+'\\Welcome_Checks.parquet')

In [ ]:
# not working, have to figure out how to not overwrite the xlsx
excel_file = working_path+'\\J7099_SIE_Alterung_Results_SE.xlsx'
worksheet_name="Welcome Checks"
with pd.ExcelFile(excel_file) as xls:
    worksheet = xls.parse(worksheet_name)
    last_row = worksheet.shape[0]
    last_col = worksheet.shape[1]
df_results.to_excel(excel_file, sheet_name=worksheet_name, index=False, header=False, startrow=5, startcol=4)

df_results=df_results.sort_index()
df_results.to_excel(working_path+'\\Measurements_SE.xlsx',index=False, startrow=5,startcol=4)
#df_results.to_csv(working_path+'\\results_siemens.csv',index=False, float_format='%.9f', decimal='.')